# 01 EDA: разведочный анализ логов Ozon Similar Products

Цель: понять, готовы ли текущие логи к preprocessing, sessionization и первому co-visitation baseline.

## 1. Импорты и пути

Таблица пользовательских действий большая, поэтому ноутбук не загружает весь лог в память. Используем parquet metadata, lazy scans, выбор нужных колонок и дневные проходы для дорогих проверок.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT

WindowsPath('C:/Users/User/PycharmProjects/OZON-Similar-products/pupupu')

In [3]:
import polars as pl

from ozon_similar_products.data import load_products
from ozon_similar_products.data.loaders import (
    find_parquet_payload_dir,
    get_path_from_config,
    load_configs,
)
from ozon_similar_products.data.profiling import (
    action_profile,
    null_profile,
    parquet_dataset_overview,
    parquet_partition_profile,
    partition_row_counts,
    schema_overview,
)
from ozon_similar_products.data.schemas import KNOWN_ACTION_TYPES
from ozon_similar_products.features.item_popularity import (
    load_event_weights,
    weighted_item_popularity,
)
from ozon_similar_products.preprocessing.session_checks import time_diff_summary

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(30)

PROJECT_CONFIG = load_configs(project_root=PROJECT_ROOT)
DATA_CONFIG = PROJECT_CONFIG["data"]

PRODUCTS_BASE_DIR = get_path_from_config(
    PROJECT_CONFIG,
    section="data",
    key="product_information_dir",
)
USER_ACTIONS_BASE_DIR = get_path_from_config(
    PROJECT_CONFIG,
    section="data",
    key="user_actions_dir",
)

PRODUCTS_ROOT = find_parquet_payload_dir(
    base_dir=PRODUCTS_BASE_DIR,
    payload_root_names=DATA_CONFIG["product_information"]["payload_root_names"],
    parquet_glob=DATA_CONFIG["product_information"]["parquet_glob"],
)
ACTIONS_ROOT = find_parquet_payload_dir(
    base_dir=USER_ACTIONS_BASE_DIR,
    payload_root_names=DATA_CONFIG["user_actions"]["payload_root_names"],
    parquet_glob=DATA_CONFIG["user_actions"]["parquet_glob"],
)

PRODUCTS_ROOT, ACTIONS_ROOT

(WindowsPath('C:/Users/User/PycharmProjects/OZON-Similar-products/pupupu/data/raw/product_information'),
 WindowsPath('C:/Users/User/PycharmProjects/OZON-Similar-products/pupupu/data/raw/user_actions'))

## 2. Обзор справочника товаров

Справочник товаров достаточно маленький, чтобы загрузить его полностью. Проверяем `item_id`, полноту метаданных и пригодность признаков для будущего fallback и business rules.

In [4]:
products = load_products(config=PROJECT_CONFIG)

product_schema = schema_overview(products)
product_nulls = null_profile(products)
product_summary = pl.DataFrame(
    {
        "метрика": [
            "строк",
            "уникальных item_id",
            "строк с дублями item_id",
            "уникальных category_id",
            "уникальных category_name",
            "уникальных brand",
            "уникальных type",
        ],
        "значение": [
            products.height,
            products.select(pl.col("item_id").n_unique()).item(),
            products.height - products.select(pl.col("item_id").n_unique()).item(),
            products.select(pl.col("category_id").n_unique()).item(),
            products.select(pl.col("category_name").n_unique()).item(),
            products.select(pl.col("brand").n_unique()).item(),
            products.select(pl.col("type").n_unique()).item(),
        ],
    }
)

product_summary, product_schema, product_nulls

(shape: (7, 2)
 ┌──────────────────────────┬──────────┐
 │ метрика                  ┆ значение │
 │ ---                      ┆ ---      │
 │ str                      ┆ i64      │
 ╞══════════════════════════╪══════════╡
 │ строк                    ┆ 130035   │
 │ уникальных item_id       ┆ 130035   │
 │ строк с дублями item_id  ┆ 0        │
 │ уникальных category_id   ┆ 834      │
 │ уникальных category_name ┆ 796      │
 │ уникальных brand         ┆ 6682     │
 │ уникальных type          ┆ 2184     │
 └──────────────────────────┴──────────┘,
 shape: (6, 2)
 ┌───────────────┬────────┐
 │ column        ┆ dtype  │
 │ ---           ┆ ---    │
 │ str           ┆ str    │
 ╞═══════════════╪════════╡
 │ item_id       ┆ Int32  │
 │ name          ┆ String │
 │ brand         ┆ String │
 │ type          ┆ String │
 │ category_id   ┆ Int32  │
 │ category_name ┆ String │
 └───────────────┴────────┘,
 shape: (6, 4)
 ┌───────────────┬───────────┬────────────┬────────────┐
 │ column        ┆ row_coun

In [5]:
top_categories = (
    products.group_by(["category_id", "category_name"])
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)

top_brands = (
    products.group_by("brand")
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)

top_types = (
    products.group_by("type")
    .agg(pl.len().alias("items"))
    .sort("items", descending=True)
    .head(30)
)

top_categories, top_brands, top_types

(shape: (30, 3)
 ┌─────────────┬─────────────────────────────────┬───────┐
 │ category_id ┆ category_name                   ┆ items │
 │ ---         ┆ ---                             ┆ ---   │
 │ i32         ┆ str                             ┆ u32   │
 ╞═════════════╪═════════════════════════════════╪═══════╡
 │ 54          ┆ Наполнители                     ┆ 7698  │
 │ 445         ┆ Сухие корма                     ┆ 3287  │
 │ 347         ┆ Для цветного белья              ┆ 3142  │
 │ 754         ┆ Сухие корма                     ┆ 2954  │
 │ 771         ┆ Влажные корма                   ┆ 2515  │
 │ 671         ┆ Блочные                         ┆ 2211  │
 │ 661         ┆ Специальные чистящие средства   ┆ 1939  │
 │ 731         ┆ Подгузники-трусики              ┆ 1860  │
 │ 849         ┆ Зубные пасты, гели              ┆ 1805  │
 │ 301         ┆ Смартфоны                       ┆ 1739  │
 │ 817         ┆ Для белого белья                ┆ 1538  │
 │ 284         ┆ Электрические чайники  

## 3. Логи действий: parquet metadata и дневные партиции

Этот блок читает parquet metadata, а не сами строки. Он должен выполняться быстро даже на полном датасете.

In [6]:
actions_overview = parquet_dataset_overview(ACTIONS_ROOT)
actions_file_profile = parquet_partition_profile(ACTIONS_ROOT)
date_action_counts = partition_row_counts(ACTIONS_ROOT)

date_summary = (
    date_action_counts.group_by("date")
    .agg(
        pl.col("rows").sum().alias("rows"),
        pl.col("files").sum().alias("files"),
        pl.col("file_size_bytes").sum().alias("file_size_bytes"),
    )
    .sort("date")
)

action_rows_from_metadata = (
    date_action_counts.group_by("action_type")
    .agg(pl.col("rows").sum().alias("rows"))
    .with_columns((pl.col("rows") / pl.col("rows").sum()).alias("share"))
    .sort("rows", descending=True)
)

actions_overview, date_summary.head(), date_summary.tail(), action_rows_from_metadata

(shape: (1, 3)
 ┌───────┬───────────┬─────────────────┐
 │ files ┆ rows      ┆ file_size_bytes │
 │ ---   ┆ ---       ┆ ---             │
 │ u32   ┆ i64       ┆ i64             │
 ╞═══════╪═══════════╪═════════════════╡
 │ 305   ┆ 561836992 ┆ 11761209986     │
 └───────┴───────────┴─────────────────┘,
 shape: (5, 4)
 ┌────────────┬──────────┬───────┬─────────────────┐
 │ date       ┆ rows     ┆ files ┆ file_size_bytes │
 │ ---        ┆ ---      ┆ ---   ┆ ---             │
 │ str        ┆ i64      ┆ u32   ┆ i64             │
 ╞════════════╪══════════╪═══════╪═════════════════╡
 │ 2024-03-01 ┆ 10446059 ┆ 5     ┆ 223224248       │
 │ 2024-03-02 ┆ 11454573 ┆ 5     ┆ 244129508       │
 │ 2024-03-03 ┆ 11127161 ┆ 5     ┆ 238471083       │
 │ 2024-03-04 ┆ 9638934  ┆ 5     ┆ 208663397       │
 │ 2024-03-05 ┆ 10859997 ┆ 5     ┆ 233281986       │
 └────────────┴──────────┴───────┴─────────────────┘,
 shape: (5, 4)
 ┌────────────┬─────────┬───────┬─────────────────┐
 │ date       ┆ rows    ┆ files

In [8]:
partition_health = pl.DataFrame(
    {
        "метрика": [
            "минимальная дата",
            "максимальная дата",
            "дней",
            "parquet-файлов",
            "строк",
            "минимум строк за день",
            "максимум строк за день",
            "найденные известные action_type",
        ],
        "значение": [
            str(date_summary.select(pl.col("date").min()).item()),
            str(date_summary.select(pl.col("date").max()).item()),
            str(date_summary.height),
            str(actions_overview["files"].item()),
            str(actions_overview["rows"].item()),
            str(date_summary.select(pl.col("rows").min()).item()),
            str(date_summary.select(pl.col("rows").max()).item()),
            ", ".join(sorted(set(action_rows_from_metadata["action_type"].to_list()) & set(KNOWN_ACTION_TYPES))),
        ],
    }
)

partition_health

метрика,значение
str,str
"""минимальная дата""","""2024-03-01"""
"""максимальная дата""","""2024-04-30"""
"""дней""","""61"""
"""parquet-файлов""","""305"""
"""строк""","""561836992"""
"""минимум строк за день""","""7431439"""
"""максимум строк за день""","""12215617"""
"""найденные известные action_typ…","""click, favorite, search, to_ca…"


## 4. Lazy scan и схема логов

Дальше используем lazy scans и выбираем только те колонки, которые нужны для конкретной проверки.

In [9]:
user_actions_lf = pl.scan_parquet(str(ACTIONS_ROOT / "**" / "*.parquet"), hive_partitioning=True)
actions_schema = schema_overview(user_actions_lf)
actions_schema

column,dtype
str,str
"""user_id""","""Int32"""
"""date""","""Date"""
"""timestamp""","""Datetime(time_unit='ns', time_…"
"""action_type""","""String"""
"""widget_name""","""String"""
"""search_query""","""String"""
"""item_id""","""Int32"""


## 5. Анализ типов событий и пропусков

Этот проход читает только ключевые колонки. У событий search поле item_id может быть пустым; это нормально, такие события нужно обрабатывать отдельно от прямых товарных взаимодействий.

In [ ]:
key_action_columns = ["user_id", "timestamp", "action_type", "search_query", "item_id"]
actions_key_lf = user_actions_lf.select(key_action_columns)

actions_by_type = action_profile(actions_key_lf).rename(
    {"users": "unique_users", "items": "unique_items"}
)
actions_nulls = null_profile(actions_key_lf)

actions_by_type, actions_nulls

In [ ]:
missing_value_decisions = pl.DataFrame(
    {
        "условие": [
            "action_type = search and item_id is null",
            "action_type in view/click/favorite/to_cart and item_id is null",
            "user_id is null",
            "timestamp is null",
            "action_type is null",
            "search_query is null for search",
        ],
        "решение": [
            "оставить вне item graph",
            "удалить для графа",
            "удалить для sessionization",
            "удалить для последовательностей",
            "удалить для взвешенных сигналов",
            "оставить событие, но исключить из источника co-search кандидатов",
        ],
        "причина": [
            "search описывает намерение, а не прямое товарное взаимодействие",
            "для item-to-item пар нужны два конкретных товара",
            "нельзя построить пользовательскую историю",
            "нельзя надежно упорядочить события",
            "нельзя сопоставить событие с весом baseline",
            "текст запроса нужен для будущих query-intent сигналов",
        ],
    }
)

missing_value_decisions

## 6. Анализ пользователей

Этот блок оценивает активность пользователей, чтобы найти bot-like или технический трафик. Используем полный лог, но читаем только user_id, timestamp, action_type и item_id.

In [ ]:
direct_action_types = ["view", "click", "favorite", "to_cart"]
direct_item_events_lf = (
    user_actions_lf
    .filter(pl.col("action_type").is_in(direct_action_types))
    .filter(pl.col("item_id").is_not_null())
    .select(["user_id", "date", "timestamp", "action_type", "item_id"])
)

user_activity_lf = direct_item_events_lf.group_by("user_id").agg(
    pl.len().alias("events"),
    pl.col("item_id").n_unique().alias("unique_items"),
    pl.col("date").n_unique().alias("active_days"),
    pl.col("timestamp").min().alias("first_event_at"),
    pl.col("timestamp").max().alias("last_event_at"),
)

user_activity_summary = user_activity_lf.select(
    pl.len().alias("users"),
    pl.col("events").sum().alias("events"),
    pl.col("events").mean().alias("events_mean"),
    pl.col("events").quantile(0.5).alias("events_p50"),
    pl.col("events").quantile(0.9).alias("events_p90"),
    pl.col("events").quantile(0.95).alias("events_p95"),
    pl.col("events").quantile(0.99).alias("events_p99"),
    pl.col("events").max().alias("events_max"),
).collect()

top_active_users = user_activity_lf.sort("events", descending=True).limit(50).collect()

user_activity_summary, top_active_users

## 7. Анализ товаров и popularity bias

Этот блок считает взвешенную популярность товаров, но не строит рекомендации. `search` по умолчанию исключается.

In [ ]:
event_weights = load_event_weights()
item_popularity_top = weighted_item_popularity(
    user_actions_lf.select(["user_id", "timestamp", "action_type", "item_id"]),
    event_weights=event_weights,
    top_n=100,
)

item_coverage = direct_item_events_lf.select(
    pl.col("item_id").n_unique().alias("items_with_behavior")
).collect()

catalog_behavior_coverage = item_coverage.with_columns(
    pl.lit(products.height).alias("catalog_items"),
    (pl.col("items_with_behavior") / pl.lit(products.height)).alias("catalog_behavior_coverage"),
)

item_popularity_top, catalog_behavior_coverage

In [ ]:
popularity_decisions = pl.DataFrame(
    {
        "вопрос": [
            "Использовать raw pair_count как финальный score?",
            "Добавлять min_pair_count?",
            "Добавлять min_unique_users?",
            "Нормализовать популярность уже в baseline?",
            "Нужен fallback для редких и новых товаров?",
        ],
        "решение": [
            "нет",
            "да, начать с config min_pair_count=2",
            "да, добавить после появления pair statistics",
            "да, cosine-like normalization в первом baseline",
            "да, fallback по category/type/brand из товарных метаданных",
        ],
        "причина": [
            "популярные товары доминируют в сырой совместной встречаемости",
            "одиночные co-events слишком шумные",
            "один очень активный пользователь не должен определять похожесть",
            "распределение событий сильно смещено в сторону популярных товаров",
            "для long-tail товаров поведенческие сигналы отсутствуют или слабы",
        ],
    }
)

popularity_decisions

## 8. Анализ поисковых запросов

`search` не входит в первый прямой item-to-item graph, но позже может стать источником co-search сигнала.

In [ ]:
search_lf = user_actions_lf.filter(pl.col("action_type") == "search")

search_summary = search_lf.select(
    pl.len().alias("search_events"),
    pl.col("user_id").n_unique().alias("search_users"),
    pl.col("search_query").drop_nulls().n_unique().alias("unique_queries"),
    pl.col("search_query").null_count().alias("empty_query_rows"),
).collect().with_columns(
    (pl.col("empty_query_rows") / pl.col("search_events")).alias("empty_query_share")
)

top_queries = (
    search_lf.filter(pl.col("search_query").is_not_null())
    .group_by("search_query")
    .agg(pl.len().alias("rows"), pl.col("user_id").n_unique().alias("users"))
    .sort("rows", descending=True)
    .limit(50)
    .collect()
)

search_summary, top_queries

## 9. Timestamp и возможность собрать сессии

Полная проверка session feasibility намеренно считается по дням. Так мы избегаем одного огромного глобального sort, но всё равно проходим по всем датам.

In [ ]:
session_timeout_minutes = 30
date_values = date_summary["date"].to_list()
session_summaries = []

for index, date_value in enumerate(date_values, start=1):
    print(f"[{index}/{len(date_values)}] session feasibility: {date_value}")
    day_path = ACTIONS_ROOT / f"date={date_value}"
    day_lf = (
        pl.scan_parquet(str(day_path / "**" / "*.parquet"), hive_partitioning=True)
        .filter(pl.col("action_type").is_in(direct_action_types))
        .filter(pl.col("item_id").is_not_null())
        .select(["user_id", "timestamp", "item_id"])
    )
    day_summary = time_diff_summary(
        day_lf,
        timeout_minutes=session_timeout_minutes,
        group_cols="user_id",
    ).with_columns(pl.lit(date_value).alias("date"))
    session_summaries.append(day_summary)

session_feasibility = pl.concat(session_summaries).select(
    ["date", "events", "time_diffs", "sessions", "gaps_over_timeout", "p50_seconds", "p75_seconds", "p90_seconds", "p95_seconds", "p99_seconds"]
)

session_feasibility

In [ ]:
session_decisions = pl.DataFrame(
    {
        "вопрос": [
            "Можно ли использовать timestamp для user timelines?",
            "Начальный session timeout",
            "Max items per session",
            "Как обрабатывать очень длинные сессии",
            "Как обрабатывать экстремально активных пользователей",
        ],
        "решение": [
            "да, если проверки null/parse остаются чистыми",
            "30 минут",
            "начать с config max_items_per_session=50",
            "разбивать по timeout и ограничивать число товаров до генерации пар",
            "пометить p99+ пользователей и рассмотреть дневные cap-ограничения в preprocessing",
        ],
        "причина": [
            "co-visitation требует упорядоченных действий пользователя",
            "стандартное MVP-значение, которое легко объяснить",
            "предотвращает комбинаторный взрыв числа пар",
            "длинные сессии смешивают разные намерения",
            "один аномальный пользователь может создать много шумных пар",
        ],
    }
)

session_decisions

## 10. Чеклист готовности baseline и выводы EDA

Заполняем этот блок после выполнения ноутбука. Таблица становится мостиком от EDA к preprocessing и co-visitation baseline.

In [ ]:
baseline_readiness = pl.DataFrame(
    {
        "проверка": [
            "данные загружены",
            "схема совпадает с контрактом",
            "timestamp корректно парсится",
            "период данных понятен",
            "дневные партиции доступны",
            "типы событий понятны",
            "типы событий для графа выбраны",
            "правило обработки search определено",
            "пропуски классифицированы",
            "аномалии пользователей проанализированы",
            "popularity bias проанализирован",
            "sessionization возможна",
            "можно переходить к preprocessing",
            "можно переходить к co-visitation baseline",
        ],
        "статус": [
            "готово",
            "готово, если actions_schema совпадает с ожидаемыми колонками",
            "готово, если timestamp имеет тип Datetime и допустимую долю пропусков",
            "готово",
            "готово",
            "готово",
            "готово: view/click/favorite/to_cart",
            "готово: хранить отдельно для будущего co-search",
            "готово после просмотра action/null таблиц",
            "готово после просмотра user_activity_summary",
            "готово после просмотра item_popularity_top",
            "готово после просмотра session_feasibility",
            "да, после фиксации thresholds",
            "да, после реализации preprocessing decisions",
        ],
    }
)

baseline_readiness

## EDA conclusions

1. Product information
   - Количество товаров: ...
   - Уникальных `item_id`: ...
   - Дубли `item_id`: ...
   - Пропуски в метаданных: ...
   - Можно ли использовать категорию/бренд для fallback и business rules: ...

2. User actions
   - Количество строк: ...
   - Период данных: ...
   - Количество пользователей: ...
   - Количество товаров с поведением: ...
   - Дневные аномалии: ...

3. Action types
   - Для co-visitation используем: `view`, `click`, `favorite`, `to_cart`.
   - `search` храним отдельно для будущего co-search сигнала.

4. Missing values
   - `search` без `item_id` считаем допустимым.
   - Товарные события без `item_id` удаляем при построении графа.
   - Критичные пропуски: ...

5. Users
   - Подозрение на bot-like или технический трафик: ...
   - Решение по p99+ пользователям и дневным cap-ограничениям: ...

6. Items
   - Есть ли popularity bias: ...
   - Как определяем long-tail: ...
   - Нужен ли fallback: ...

7. Sessions
   - Рекомендуемый timeout: 30 минут, если профиль time gaps не покажет обратное.
   - Ограничение длины сессии: начать с `max_items_per_session=50`.

8. Baseline decisions
   - Первый baseline: weighted co-visitation graph.
   - Score: weighted pair score + cosine-like popularity normalization.
   - Минимальные quality gates: `min_pair_count`, затем `min_unique_users`.
